# 👈👉 Python Two Pointer — The Master Guide
### *From Zero to Interview-Ready*

---

> **Mental Model First:**
> Two pointers are like two fingers tracing a ruler from both ends.
> You start one finger at the left and one at the right.
> After each step you move whichever finger gives you more information.
> The key rule: you never move a finger backward. One pass, O(n).

---

## 📋 Table of Contents

| # | Section |
|---|---------|
| 1 | [What Is Two Pointer? The Visual Model](#1) |
| 2 | [Setup / Initialization](#2) |
| 3 | [The Core API — All Operations](#3) |
| 4 | [Decision Map — When To Use What](#4) |
| 5 | [Pattern 1: Opposite Ends — LC 167 Two Sum II](#5) |
| 6 | [Pattern 2: Three Sum — LC 15](#6) |
| 7 | [Pattern 3: Trapping Rain Water — LC 42](#7) |
| 8 | [Pattern 4: Container With Most Water — LC 11](#8) |
| 9 | [Pattern 5: Dutch Flag Partition — LC 75](#9) |
| 10 | [The Two Pointer Decision Map](#10) |
| 11 | [Interview Cheat Sheet](#11) |
| 12 | [Summary Map](#12) |

<a id='1'></a>

## 1. What Is Two Pointer? The Visual Model

```
OPPOSITE ENDS — squeeze inward

  nums = [1, 3, 5, 7, 9, 11]   target = 12

   lo=0                 hi=5
    │                    │
    ▼                    ▼
  [ 1,  3,  5,  7,  9, 11 ]
    sum = 1+11 = 12  → FOUND

  If sum < target  → move lo right  (need bigger left)
  If sum > target  → move hi left   (need smaller right)
  If sum == target → answer

THREE-POINTER PARTITION (Dutch Flag)

  lo=0  mid=0                hi=5
   │     │                    │
   ▼     ▼                    ▼
  [ 2,  0,  2,  1,  0,  1 ]
    └─ red ─┘└─ white ─┘└─ blue ─┘
     (0s)     (1s)        (2s)

  mid scans forward:
    0 → swap with lo, advance both lo and mid
    1 → advance mid only
    2 → swap with hi, retreat hi (don't advance mid — new element unseen)

WHY TWO POINTER IS O(n):
  Each pointer moves at most n steps total.
  lo+hi together: at most 2n moves → O(n)
  Never revisit a position — that is the invariant.
```

<a id='2'></a>

## 2. Setup / Initialization

In [ ]:
# Two pointer setup — all you need are two index variables
nums = [1, 3, 5, 7, 9, 11]

# OPPOSITE ENDS pattern
lo, hi = 0, len(nums) - 1          # start at both ends
print(f"opposite ends: lo={lo} → nums[lo]={nums[lo]}, hi={hi} → nums[hi]={nums[hi]}")

# SAME DIRECTION pattern (fast/slow)
slow, fast = 0, 0                   # both start at left, fast races ahead
print(f"same direction: slow={slow}, fast={fast}")

# SAME DIRECTION with offset (find pairs with gap)
left, right = 0, 1                  # right is one step ahead
print(f"offset start: left={left}, right={right}")

# THREE POINTER (partition / Dutch flag)
lo2, mid2, hi2 = 0, 0, len(nums) - 1
print(f"three pointer: lo={lo2}, mid={mid2}, hi={hi2}")

print("\nAll pointer patterns initialized.")

<a id='3'></a>

## 3. The Core API — All Operations

```
OPERATION                   COMPLEXITY   WHAT IT DOES
────────────────────────────────────────────────────────────────
lo += 1                     O(1)         advance left pointer
hi -= 1                     O(1)         retreat right pointer
nums[lo], nums[hi] = ...    O(1)         swap two elements
while lo < hi               O(n)         squeeze until pointers meet
while fast < len(nums)      O(n)         fast pointer scan

THINGS YOU DO NOT DO:
❌  Use two pointer on unsorted arrays for sum problems (sort first or use HashMap)
❌  Move both pointers on the same iteration without reason
❌  Use two pointer when you need all pairs — that is O(n²) territory
❌  Forget to handle duplicates in 3Sum (skip while nums[i] == nums[i-1])
❌  Mix up lo < hi  vs  lo <= hi  — meeting vs crossing
```

In [ ]:
# Live demo: opposite-ends squeeze on sorted array
nums = [1, 3, 5, 7, 9, 11]
target = 12
lo, hi = 0, len(nums) - 1

print("Squeeze trace — target=12")
while lo < hi:
    s = nums[lo] + nums[hi]
    print(f"  lo={lo}({nums[lo]}) hi={hi}({nums[hi]}) sum={s}", end=" → ")
    if s == target:
        print("FOUND")
        break
    elif s < target:
        print("move lo right")
        lo += 1              # sum too small — need bigger left
    else:
        print("move hi left")
        hi -= 1              # sum too large — need smaller right

print()

# Demo: swap using three-pointer
arr = [2, 0, 1, 2, 0, 1]
lo2, mid2, hi2 = 0, 0, len(arr) - 1
print("Dutch flag trace:")
while mid2 <= hi2:
    if arr[mid2] == 0:
        arr[lo2], arr[mid2] = arr[mid2], arr[lo2]  # red goes left
        lo2 += 1
        mid2 += 1
    elif arr[mid2] == 1:
        mid2 += 1                                   # white stays, advance scanner
    else:
        arr[mid2], arr[hi2] = arr[hi2], arr[mid2]  # blue goes right
        hi2 -= 1                                    # don't advance mid — new elem unseen
print(f"  result: {arr}")
print("Core API demo done.")

<a id='4'></a>

## 4. Decision Map — When To Use What

```
SIGNAL IN THE PROBLEM                  WHICH PATTERN
──────────────────────────────────────────────────────────────
sorted array + find pair with target   opposite ends squeeze
find triplet summing to zero           sort + fix one + squeeze
water / area between bars              move shorter wall
partition by value (0/1/2)             three pointers, Dutch flag
remove duplicates in-place             slow writes, fast reads
palindrome check                       lo/hi squeeze inward
cycle detection in linked list         fast (2x) + slow (1x)
```

<a id='5'></a>

## 5. 🧩 Pattern 1: Opposite Ends — LC 167 Two Sum II

---

```
PROBLEM:
  numbers is 1-indexed sorted array. Find two numbers summing to target.

TRICK:
  Sorted array means lo+hi sum is predictable.
  Too small → move lo right. Too large → move hi left.
  One pointer always moves → at most n steps total.

SLOW MOTION TRACE on numbers=[2,7,11,15], target=9:
  step  lo  hi  sum    action
   1     0   3  2+15=17  > 9 → hi--
   2     0   2  2+11=13  > 9 → hi--
   3     0   1  2+7=9   == 9 → RETURN [1,2]

KEY INSIGHT:
  Sorted array turns "find pair" from O(n²) into O(n).
  Each move eliminates one element permanently.

TIME:  O(n) — one pass, each pointer moves at most n steps
SPACE: O(1) — only two index variables
```

In [ ]:
from typing import List

def two_sum_ii(numbers: List[int], target: int) -> List[int]:
    """
    LC 167 — Two Sum II - Input Array Is Sorted
    Approach: opposite-ends squeeze on sorted array.
    Args:
        numbers (List[int]): 1-indexed sorted array, length >= 2.
        target (int): desired sum.
    Returns:
        List[int]: 1-indexed [left, right] positions of the two numbers.
    Time:  O(n) — each pointer moves at most n steps total
    Space: O(1) — two index variables only
    """
    lo, hi = 0, len(numbers) - 1   # start at both ends of sorted array

    while lo < hi:                  # stop when pointers meet — exhausted all pairs
        s = numbers[lo] + numbers[hi]
        if s == target:
            return [lo + 1, hi + 1]  # 1-indexed answer
        elif s < target:
            lo += 1                  # sum too small — left element must grow
        else:
            hi -= 1                  # sum too large — right element must shrink

    return []                        # guaranteed to find, but satisfy return type

# Slow motion on numbers=[2,7,11,15], target=9:
# step  lo  hi  sum     action
#   1    0   3  17      17>9 → hi-- → hi=2
#   2    0   2  13      13>9 → hi-- → hi=1
#   3    0   1   9       9==9 → return [1,2]

def test_harness(fn):
    tests = [
        ([2, 7, 11, 15], 9,  [1, 2]),
        ([2, 3, 4],      6,  [1, 3]),
        ([-1, 0],       -1,  [1, 2]),
        ([1, 2, 3, 4, 5], 9, [4, 5]),
        ([1, 3, 4, 5, 7, 11], 9, [3, 4]),
    ]
    passed = 0
    for *inputs, expected in tests:
        got = fn(*inputs)
        status = "PASSED" if got == expected else "FAILED"
        if status == "FAILED":
            print(f"{status} | input={inputs} | expected={expected} | got={got}")
        passed += (got == expected)
    print(f"{passed}/{len(tests)} tests passed")

test_harness(two_sum_ii)
print("two_sum_ii defined.")

<a id='6'></a>

## 6. 🧩 Pattern 2: Three Sum — LC 15

---

```
PROBLEM:
  Find all unique triplets in nums that sum to zero.

TRICK:
  Sort first. Fix one element (i). Squeeze lo/hi on the rest.
  Skip duplicates at each level to avoid duplicate triplets.

SLOW MOTION TRACE on nums=[-4,-1,-1,0,1,2]:
  i=0  fix=-4  lo=1(-1) hi=5(2)  sum=-3 → lo++
  i=0  fix=-4  lo=2(-1) hi=5(2)  sum=-3 → lo++
  i=0  fix=-4  lo=3(0)  hi=5(2)  sum=-2 → lo++
  i=0  fix=-4  lo=4(1)  hi=5(2)  sum=-1 → lo++
  i=0  done (lo>=hi)
  i=1  fix=-1  lo=2(-1) hi=5(2)  sum=0  → add [-1,-1,2], skip dups
  i=1  fix=-1  lo=3(0)  hi=4(1)  sum=0  → add [-1,0,1], skip dups
  i=2  skip (nums[2]==nums[1])
  i=3  fix=0   lo=4(1)  hi=5(2)  sum=3  → hi--
  i=3  done
  result: [[-1,-1,2],[-1,0,1]]

KEY INSIGHT:
  Sorting makes duplicate-skipping trivial — identical values are adjacent.
  The outer loop O(n) × inner squeeze O(n) = O(n²) total.

TIME:  O(n²) — outer loop O(n), inner squeeze O(n)
SPACE: O(n) — sorting is in-place but output list can be O(n²) worst case
```

In [ ]:
def three_sum(nums: List[int]) -> List[List[int]]:
    """
    LC 15 — 3Sum
    Approach: sort, fix one element, squeeze lo/hi on remainder.
    Args:
        nums (List[int]): integer array, may have duplicates.
    Returns:
        List[List[int]]: all unique triplets summing to zero.
    Time:  O(n²) — O(n log n) sort + O(n²) two-pointer sweep
    Space: O(n)  — sorting space; output not counted
    """
    nums.sort()                          # sorting enables duplicate skipping
    result = []

    for i in range(len(nums) - 2):       # fix the leftmost of the three
        if i > 0 and nums[i] == nums[i - 1]:
            continue                      # skip duplicate fixed element
        if nums[i] > 0:
            break                         # sorted → if fixed > 0, sum can't be 0

        lo, hi = i + 1, len(nums) - 1
        while lo < hi:
            s = nums[i] + nums[lo] + nums[hi]
            if s == 0:
                result.append([nums[i], nums[lo], nums[hi]])
                while lo < hi and nums[lo] == nums[lo + 1]:
                    lo += 1              # skip duplicate lo values
                while lo < hi and nums[hi] == nums[hi - 1]:
                    hi -= 1              # skip duplicate hi values
                lo += 1
                hi -= 1
            elif s < 0:
                lo += 1                  # need bigger left
            else:
                hi -= 1                  # need smaller right

    return result

# Slow motion on [-4,-1,-1,0,1,2] sorted:
# i=1(fix=-1): lo=2(-1),hi=5(2) sum=0 → found [-1,-1,2]
# i=1(fix=-1): lo=3(0), hi=4(1) sum=0 → found [-1,0,1]
# i=2 skipped (duplicate of i=1)

def test_harness_3sum(fn):
    def norm(res):
        return sorted(tuple(sorted(t)) for t in res)
    tests = [
        ([-1, 0, 1, 2, -1, -4], [[-1, -1, 2], [-1, 0, 1]]),
        ([0, 1, 1],              []),
        ([0, 0, 0],              [[0, 0, 0]]),
        ([-2, 0, 0, 2, 2],      [[-2, 0, 2]]),
        ([-4, -1, -1, 0, 1, 2], [[-1, -1, 2], [-1, 0, 1]]),
    ]
    passed = 0
    for *inputs, expected in tests:
        got = fn(*inputs)
        status = "PASSED" if norm(got) == norm(expected) else "FAILED"
        if status == "FAILED":
            print(f"{status} | input={inputs} | expected={expected} | got={got}")
        passed += (norm(got) == norm(expected))
    print(f"{passed}/{len(tests)} tests passed")

test_harness_3sum(three_sum)
print("three_sum defined.")

<a id='7'></a>

## 7. 🧩 Pattern 3: Trapping Rain Water — LC 42

---

```
PROBLEM:
  Given elevation map heights[], compute total water trapped after rain.

TRICK:
  Water at position i = min(max_left[i], max_right[i]) - height[i].
  Two pointer: track max_left and max_right as you squeeze inward.
  Move whichever side has the SMALLER max wall — that side is the bottleneck.

SLOW MOTION TRACE on heights=[0,1,0,2,1,0,1,3,2,1,2,1]:
  ASCII bar chart:
    3       |       |
    2   |   | | | | |
    1 | | | | | | | | | | | |
      0 1 2 3 4 5 6 7 8 9 ...

  lo=0(h=0) hi=11(h=1) max_l=0 max_r=1
    max_l(0) < max_r(1) → process lo side
    water += max(0,0-0)=0, lo++
  lo=1(h=1) max_l=1: max_l(1) >= max_r(1) → process hi
    water += max(0,1-1)=0, hi--
  ... continues until lo meets hi
  final water = 6

KEY INSIGHT:
  Always process the side with the smaller max-wall — that wall is the
  binding constraint. You can compute water without knowing the other side.

TIME:  O(n) — one pass, each pointer moves at most n steps
SPACE: O(1) — two pointers, two running maxima
```

In [ ]:
def trap(height: List[int]) -> int:
    """
    LC 42 — Trapping Rain Water
    Approach: two pointers tracking max-left / max-right walls.
    Args:
        height (List[int]): non-negative integers, elevation map.
    Returns:
        int: total units of trapped water.
    Time:  O(n) — one pass, each pointer advances at most n steps
    Space: O(1) — only lo, hi, max_l, max_r variables
    """
    if not height:
        return 0

    lo, hi = 0, len(height) - 1
    max_l = max_r = 0
    water = 0

    while lo < hi:
        if height[lo] < height[hi]:      # left wall is the bottleneck
            if height[lo] >= max_l:
                max_l = height[lo]        # new max wall — no water here
            else:
                water += max_l - height[lo]  # water = wall height - ground
            lo += 1
        else:                             # right wall is the bottleneck
            if height[hi] >= max_r:
                max_r = height[hi]        # new max wall — no water here
            else:
                water += max_r - height[hi]  # water = wall height - ground
            hi -= 1

    return water

# Slow motion on [0,1,0,2,1,0,1,3,2,1,2,1]:
# lo=0(0), hi=11(1): 0<1 → left side. max_l=0, h=0 → max_l=0, water+=0, lo++
# lo=1(1), hi=11(1): 1>=1 → right side. h=1 → max_r=1, water+=0, hi--
# lo=1(1), hi=10(2): 1<2 → left side. h=1>=max_l(0) → max_l=1, lo++
# ... eventually water=6

def test_harness(fn):
    tests = [
        ([0,1,0,2,1,0,1,3,2,1,2,1], 6),
        ([4,2,0,3,2,5],             9),
        ([],                         0),
        ([3,0,3],                    3),
        ([1,2,3,4,5],                0),
        ([5,4,3,2,1],                0),
        ([3,1,2],                    1),
    ]
    passed = 0
    for *inputs, expected in tests:
        got = fn(*inputs)
        status = "PASSED" if got == expected else "FAILED"
        if status == "FAILED":
            print(f"{status} | input={inputs} | expected={expected} | got={got}")
        passed += (got == expected)
    print(f"{passed}/{len(tests)} tests passed")

test_harness(trap)
print("trap defined.")

<a id='8'></a>

## 8. 🧩 Pattern 4: Container With Most Water — LC 11

---

```
PROBLEM:
  Given heights[], find two lines forming a container with maximum water.

TRICK:
  Area = min(height[lo], height[hi]) * (hi - lo).
  Moving the taller wall can never increase area — width shrinks but height stays same.
  So always move the SHORTER wall — that's the only chance to get a bigger min.

SLOW MOTION TRACE on height=[1,8,6,2,5,4,8,3,7]:
  lo=0(1) hi=8(7) area=min(1,7)*8=8  → move lo (shorter)
  lo=1(8) hi=8(7) area=min(8,7)*7=49 → move hi (shorter)
  lo=1(8) hi=7(3) area=min(8,3)*6=18 → move hi
  lo=1(8) hi=6(8) area=min(8,8)*5=40 → move hi (tie → either)
  lo=1(8) hi=5(4) area=min(8,4)*4=16 → move hi
  lo=1(8) hi=4(5) area=min(8,5)*3=15 → move hi
  lo=1(8) hi=3(2) area=min(8,2)*2=4  → move hi
  lo=1(8) hi=2(6) area=min(8,6)*1=6  → move hi
  done. max_area=49

KEY INSIGHT:
  Moving the taller line provably cannot improve — width -1, min stays same.
  Greedy: always move shorter → no valid pair is skipped.

TIME:  O(n) — one squeeze pass
SPACE: O(1) — two pointers, one max variable
```

In [ ]:
def max_area(height: List[int]) -> int:
    """
    LC 11 — Container With Most Water
    Approach: greedy two pointers — always move the shorter wall.
    Args:
        height (List[int]): non-negative integers, line heights.
    Returns:
        int: maximum water area.
    Time:  O(n) — one pass, at most n moves total
    Space: O(1) — only lo, hi, best variables
    """
    lo, hi = 0, len(height) - 1
    best = 0

    while lo < hi:
        h = min(height[lo], height[hi])   # shorter wall is the bottleneck
        w = hi - lo                        # width between the two lines
        best = max(best, h * w)

        if height[lo] <= height[hi]:
            lo += 1   # moving shorter wall is the only chance to improve
        else:
            hi -= 1

    return best

# Slow motion on [1,8,6,2,5,4,8,3,7]:
# lo=0(1),hi=8(7): area=1*8=8,   best=8  → lo++ (shorter)
# lo=1(8),hi=8(7): area=7*7=49,  best=49 → hi-- (shorter)
# lo=1(8),hi=7(3): area=3*6=18,  best=49 → hi--
# ... final best=49

def test_harness(fn):
    tests = [
        ([1,8,6,2,5,4,8,3,7], 49),
        ([1,1],                1),
        ([4,3,2,1,4],         16),
        ([1,2,1],              2),
        ([2,3,4,5,18,17,6],   17),
    ]
    passed = 0
    for *inputs, expected in tests:
        got = fn(*inputs)
        status = "PASSED" if got == expected else "FAILED"
        if status == "FAILED":
            print(f"{status} | input={inputs} | expected={expected} | got={got}")
        passed += (got == expected)
    print(f"{passed}/{len(tests)} tests passed")

test_harness(max_area)
print("max_area defined.")

<a id='9'></a>

## 9. 🧩 Pattern 5: Dutch Flag Partition — LC 75 Sort Colors

---

```
PROBLEM:
  Sort array of 0s, 1s, 2s in-place using at most one pass.

TRICK:
  Three pointers: lo (next 0 slot), mid (scanner), hi (next 2 slot).
  mid advances only when element is 0 or 1.
  When swapping a 2 from mid to hi, don't advance mid — the new element is unseen.

SLOW MOTION TRACE on nums=[2,0,2,1,1,0]:
  lo=0 mid=0 hi=5  nums[mid]=2 → swap(mid,hi) → [0,0,2,1,1,2] hi=4
  lo=0 mid=0 hi=4  nums[mid]=0 → swap(lo,mid) → [0,0,2,1,1,2] lo=1,mid=1
  lo=1 mid=1 hi=4  nums[mid]=0 → swap(lo,mid) → [0,0,2,1,1,2] lo=2,mid=2
  lo=2 mid=2 hi=4  nums[mid]=2 → swap(mid,hi) → [0,0,1,1,2,2] hi=3
  lo=2 mid=2 hi=3  nums[mid]=1 → mid++ → mid=3
  lo=2 mid=3 hi=3  nums[mid]=1 → mid++ → mid=4 > hi → done
  result: [0,0,1,1,2,2]

KEY INSIGHT:
  The invariant: nums[0..lo-1] are 0s, nums[lo..mid-1] are 1s,
  nums[hi+1..n-1] are 2s. mid is the unsorted frontier.

TIME:  O(n) — mid advances at most n times, hi retreats at most n times
SPACE: O(1) — in-place, three index variables
```

In [ ]:
def sort_colors(nums: List[int]) -> None:
    """
    LC 75 — Sort Colors (Dutch National Flag)
    Approach: three-pointer partition. In-place, one pass.
    Args:
        nums (List[int]): array containing only 0, 1, 2. Modified in place.
    Returns:
        None — modifies nums in place.
    Time:  O(n) — mid advances at most n steps
    Space: O(1) — in-place, three index variables
    """
    lo, mid, hi = 0, 0, len(nums) - 1

    while mid <= hi:
        if nums[mid] == 0:
            nums[lo], nums[mid] = nums[mid], nums[lo]  # 0 belongs at lo
            lo += 1
            mid += 1    # element at old lo was 1 (invariant) → mid safe to advance
        elif nums[mid] == 1:
            mid += 1    # 1 is already in the right zone, just scan past
        else:           # nums[mid] == 2
            nums[mid], nums[hi] = nums[hi], nums[mid]  # 2 belongs at hi
            hi -= 1
            # don't advance mid — the element just moved to mid is unseen

# Slow motion on [2,0,2,1,1,0]:
# lo=0,mid=0,hi=5: val=2 → swap(0,5) → [0,0,2,1,1,2] hi=4
# lo=0,mid=0,hi=4: val=0 → swap(0,0) → same, lo=1,mid=1
# lo=1,mid=1,hi=4: val=0 → swap(1,1) → same, lo=2,mid=2
# lo=2,mid=2,hi=4: val=2 → swap(2,4) → [0,0,1,1,2,2] hi=3
# lo=2,mid=2,hi=3: val=1 → mid=3
# lo=2,mid=3,hi=3: val=1 → mid=4 > hi → done
# result: [0,0,1,1,2,2]

def test_harness(fn):
    import copy
    tests = [
        ([2,0,2,1,1,0],   [0,0,1,1,2,2]),
        ([2,0,1],          [0,1,2]),
        ([0],              [0]),
        ([1],              [1]),
        ([0,0,0],          [0,0,0]),
        ([2,2,2],          [2,2,2]),
        ([1,2,0,1,2,0,1], [0,0,1,1,1,2,2]),
    ]
    passed = 0
    for *inputs, expected in tests:
        inp = copy.deepcopy(inputs[0])
        fn(inp)
        status = "PASSED" if inp == expected else "FAILED"
        if status == "FAILED":
            print(f"{status} | input={inputs} | expected={expected} | got={inp}")
        passed += (inp == expected)
    print(f"{passed}/{len(tests)} tests passed")

test_harness(sort_colors)
print("sort_colors defined.")

<a id='10'></a>

## 10. The Two Pointer Decision Map

```
QUESTION TYPE                          KEY TECHNIQUE         LC PROBLEMS
─────────────────────────────────────────────────────────────────────────
Sorted array + find pair with sum      Opposite ends         167, 1
Triplets summing to zero               Sort + fix + squeeze  15
Max water / area between bars          Move shorter wall     11
Trapped water between bars             Move smaller max      42
Partition 0s/1s/2s in one pass         Dutch flag, 3 ptrs    75
Remove duplicates in-place             Slow write, fast read 26, 80
Palindrome check on string             lo/hi squeeze         125, 680
Cycle in linked list                   Fast(2x) + slow(1x)   141, 142

WHICH TEMPLATE TO REACH FOR:
  sorted + sum problem      → lo=0, hi=n-1, squeeze
  partition / in-place sort → lo, mid, hi three-pointer
  remove/overwrite in-place → slow (write head), fast (scanner)
  area / water problems     → move the constraining (shorter) side
```

<a id='11'></a>

## 11. Interview Cheat Sheet

**1. When to reach for Two Pointer:**

| Signal | What To Do |
|--------|------------|
| Sorted array + find pair/triplet | Opposite-ends squeeze |
| Max/min area between elements | Move constraining side |
| In-place partition by value | Three-pointer Dutch flag |
| Palindrome / symmetric check | lo/hi squeeze |
| Remove in-place, keep order | slow/fast write-read |

**2. Core operations — memorize these:**

```python
lo, hi = 0, len(arr) - 1    # opposite ends init
while lo < hi:               # terminate when pointers meet
    lo += 1                  # move left pointer right
    hi -= 1                  # move right pointer left
    arr[lo], arr[hi] = arr[hi], arr[lo]  # in-place swap
```

**3. Common templates:**

```python
# TEMPLATE 1: OPPOSITE ENDS (sorted array)
lo, hi = 0, len(arr) - 1
while lo < hi:
    s = arr[lo] + arr[hi]
    if s == target: return [lo, hi]
    elif s < target: lo += 1
    else: hi -= 1

# TEMPLATE 2: DUTCH FLAG (partition 3 values)
lo, mid, hi = 0, 0, len(arr) - 1
while mid <= hi:
    if arr[mid] == 0:   arr[lo], arr[mid] = arr[mid], arr[lo]; lo += 1; mid += 1
    elif arr[mid] == 1: mid += 1
    else:               arr[mid], arr[hi] = arr[hi], arr[mid]; hi -= 1

# TEMPLATE 3: THREE SUM
arr.sort()
for i in range(len(arr) - 2):
    if i > 0 and arr[i] == arr[i-1]: continue
    lo, hi = i + 1, len(arr) - 1
    while lo < hi:
        s = arr[i] + arr[lo] + arr[hi]
        if s == 0: result.append([arr[i], arr[lo], arr[hi]]); lo += 1; hi -= 1
        elif s < 0: lo += 1
        else: hi -= 1
```

**4. Gotchas:**

```
❌  Using two pointer on unsorted arrays for sum (use HashMap instead)
❌  Advancing mid after swapping a 2 in Dutch flag
❌  Forgetting to skip duplicates in 3Sum (both i and lo/hi levels)
❌  Using lo <= hi when you want lo < hi — off-by-one error
✅  Sort the array first for sum/triplet problems
✅  When in doubt about which pointer to move: move the constraining one
```

<a id='12'></a>

## 12. Summary Map

```
TWO POINTER
│
├── Opposite Ends (sorted)
│     ├── LC 167 Two Sum II
│     └── LC 15 Three Sum (+ outer fix loop)
│
├── Move Constraining Side
│     ├── LC 11 Container With Most Water
│     └── LC 42 Trapping Rain Water
│
└── Three Pointer / Partition
      └── LC 75 Sort Colors (Dutch National Flag)

CHOOSE BY:
  sorted + find sum pair    → opposite ends
  area / water / height     → move shorter/smaller side
  partition 3 value classes → lo / mid / hi

ALL PATTERNS ARE O(n) — the pointers together take at most 2n steps.
```

---
*End of Two Pointer Master Guide — Sean Edition*